In [1]:
import numpy as np
import random
from operator import attrgetter
import time

In [2]:
def objective_function(x):
    return x[0]**2+x[1]**2

## RS-SPSO — implementation

**RS-SPSO** (Respawning Speciation-based PSO). `rs_spso(obj_func, bounds, m)` returns up to `m`
distinct minima. It speciates the swarm into sub-swarms (one per target minimum), repels species
apart, polishes a stalled species with DE, and — the distinctive part — **respawns** a converged
species into unexplored space to hunt the minima still missing (instead of freezing its particles).
Building blocks below in order: particle + species, seed/assignment/repulsion/refiner helpers, the
main loop, then a Himmelblau demo (4 known equal minima) to validate it.

In [3]:
import numpy as np
import random
from scipy.optimize import differential_evolution


def euclidean(a, b):
    return np.linalg.norm(a - b)


class Particle:
    def __init__(self, bounds, obj_func):
        self.position = np.array([random.uniform(lo, hi) for lo, hi in bounds])
        self.velocity = np.zeros(len(bounds))
        self.pBestPosition = self.position.copy()
        self.pBestScore = obj_func(self.position)
        self.swarm_id = 0                     # 0 = free particle; 1..m = sub_swarm[id-1]

    def update_velocity(self, sBestPosition, repulsion, c1, c2, w):
        dim = len(self.position)
        r1 = np.random.random(dim)            # resampled every step (per-dimension)
        r2 = np.random.random(dim)
        cognitive = c1 * r1 * (self.pBestPosition - self.position)
        social    = c2 * r2 * (sBestPosition - self.position)
        self.velocity = w * self.velocity + cognitive + social + repulsion

    def update_position(self, bounds):
        self.position = self.position + self.velocity
        for d, (lo, hi) in enumerate(bounds):
            if self.position[d] < lo:
                self.position[d] = lo
                self.velocity[d] = 0.0        # kill velocity into the wall
            elif self.position[d] > hi:
                self.position[d] = hi
                self.velocity[d] = 0.0

    def evaluate(self, obj_func):
        score = obj_func(self.position)
        if score < self.pBestScore:
            self.pBestPosition = self.position.copy()
            self.pBestScore = score
        return score


class SubSwarm:
    def __init__(self, seed, index):
        self.index = index
        self.members = [seed]
        seed.swarm_id = index + 1
        self.sBestPosition = seed.pBestPosition.copy()
        self.sBestScore = seed.pBestScore
        self.stall_counter = 0
        self.active = True

    def update_sBest(self):
        prev = self.sBestScore
        for p in self.members:
            if p.pBestScore < self.sBestScore:
                self.sBestScore = p.pBestScore
                self.sBestPosition = p.pBestPosition.copy()
        return prev - self.sBestScore        # improvement this step (>= 0)

    def radius(self):
        return max(euclidean(p.position, self.sBestPosition) for p in self.members)

    def respawn(self, bounds, obj_func, found, avoid, tries=30):
        """Relocate this species' own particles into unexplored space (> avoid from every
        found optimum) to hunt for a still-missing minimum. Keeps the species active.
        This is the RS ('respawn') part: converged species explore instead of freezing."""
        for p in self.members:
            cand = None
            for _ in range(tries):
                cand = np.array([random.uniform(lo, hi) for lo, hi in bounds])
                if all(euclidean(cand, fp) > avoid for fp in found):
                    break
            p.position = cand
            p.velocity = np.zeros(len(bounds))
            p.pBestPosition = cand.copy()
            p.pBestScore = obj_func(cand)
        best = min(self.members, key=lambda p: p.pBestScore)
        self.sBestPosition = best.pBestPosition.copy()
        self.sBestScore = best.pBestScore
        self.stall_counter = 0

    def free_members(self):
        for p in self.members:
            p.swarm_id = 0
        self.members = []
        self.active = False

In [4]:
def repulsion_force(p, all_particles, k_swarm, k_particle, v_repel_max, eps=1e-12):
    """Push p away from the nearest particle of a different swarm. Bounded by v_repel_max."""
    nearest, nd = None, np.inf
    for q in all_particles:
        if q is p or q.swarm_id == p.swarm_id:
            continue
        d = euclidean(p.position, q.position)
        if d < nd:
            nd, nearest = d, q
    if nearest is None:
        return np.zeros(len(p.position))
    direction = (p.position - nearest.position) / (nd + eps)
    strength = k_swarm if nearest.swarm_id != 0 else k_particle   # swarm-swarm >> particle
    force = strength * direction / (nd ** 2 + eps)
    return np.clip(force, -v_repel_max, v_repel_max)               # cap is essential


def select_seeds(sorted_swarm, m, r):
    """Best-first seeds, each > r from all others. Farthest-point fallback if < m found."""
    seeds = [sorted_swarm[0]]
    for p in sorted_swarm[1:]:
        if all(euclidean(p.position, s.position) > r for s in seeds):
            seeds.append(p)
        if len(seeds) == m:
            break
    if len(seeds) < m:
        remaining = [p for p in sorted_swarm if p not in seeds]
        while len(seeds) < m and remaining:
            # most-isolated leftover: max over p of its distance to the nearest seed
            best_p = max(remaining,
                         key=lambda p: min(euclidean(p.position, s.position) for s in seeds))
            seeds.append(best_p)
            remaining.remove(best_p)
    return seeds


def build_subswarms(sorted_swarm, seeds, n):
    """Assign each non-seed to its nearest seed that still has capacity ('look farther')."""
    subswarms = [SubSwarm(s, i) for i, s in enumerate(seeds)]
    seed_ids = {id(s) for s in seeds}
    remaining = [p for p in sorted_swarm if id(p) not in seed_ids]
    for p in remaining:
        order = sorted(range(len(seeds)),
                       key=lambda i: euclidean(p.position, seeds[i].position))
        for i in order:
            if len(subswarms[i].members) < n:
                subswarms[i].members.append(p)
                p.swarm_id = i + 1
                break
        # if every swarm is full, p stays swarm_id = 0 (free reserve pool)
    return subswarms


def default_refiner(obj_func, x0, bounds, pad_frac=0.05):
    """Refine a stalled swarm with Differential Evolution, restricted to a LOCAL box
    around x0. Keeping DE basin-local stops every stalled swarm collapsing onto the
    single global optimum. pad_frac sets the half-width as a fraction of each range."""
    x0 = np.asarray(x0)
    local_bounds = []
    for d, (lo, hi) in enumerate(bounds):
        pad = pad_frac * (hi - lo)
        local_bounds.append((max(lo, x0[d] - pad), min(hi, x0[d] + pad)))
    res = differential_evolution(obj_func, local_bounds, x0=x0, polish=True, tol=1e-6)
    return res.x, res.fun

In [5]:
def rs_spso(obj_func, bounds, m,
            N=40, r=None, c1=1.5, c2=1.5, w=0.7,
            max_iteration=200, pos_tol=1e-4, f_tol=1e-8, patience=15,
            k_swarm=1.0, k_particle=0.1, v_repel_max=None, sep=None,
            refiner=default_refiner, verbose=True):
    """RS-SPSO -- Respawning Speciation-based PSO.

    Finds up to m distinct minima of obj_func, returned as (position, score).
    Speciated sub-swarms search in parallel, repel each other, and a stalled species
    is polished by DE. When a species converges (or is DE-refined) its optimum is
    recorded if new, then the species RESPAWNS into unexplored space to hunt for the
    minima still missing -- so converged particles keep working instead of freezing."""
    assert m <= N, "need at least as many particles as solutions"
    lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
    diag = euclidean(lo, hi)
    if r is None:
        r = 0.1 * diag                       # seed spacing ~ 10% of domain diagonal
    if v_repel_max is None:
        v_repel_max = 0.1 * diag
    if sep is None:
        sep = 0.02 * diag                    # two optima counted "the same" if closer than sep
    avoid = r * 0.5                          # respawn keeps new explorers this far from found optima
    n = max(2, N // m)                        # particles per species (1 seed + rest)

    swarm = [Particle(bounds, obj_func) for _ in range(N)]
    sorted_swarm = sorted(swarm, key=lambda p: p.pBestScore)
    seeds = select_seeds(sorted_swarm, m, r)
    subswarms = build_subswarms(sorted_swarm, seeds, n)

    t0 = time.time()
    solutions = []
    for iteration in range(max_iteration):
        active = [S for S in subswarms if S.active]
        if not active or len(solutions) >= m:
            break

        # --- move every active species ---
        for S in active:
            for p in S.members:
                rep = repulsion_force(p, swarm, k_swarm, k_particle, v_repel_max)
                p.update_velocity(S.sBestPosition, rep, c1, c2, w)
                p.update_position(bounds)
                p.evaluate(obj_func)
            improvement = S.update_sBest()
            S.stall_counter = S.stall_counter + 1 if improvement < f_tol else 0

        # --- converged / stalled -> record if new, then respawn to explore (or retire) ---
        for S in active:
            converged = S.radius() < pos_tol
            stalled = S.stall_counter >= patience
            if not (converged or stalled):
                continue
            if stalled and not converged:
                t_de = time.time()
                x, fx = refiner(obj_func, S.sBestPosition, bounds)
                cand, cscore = np.asarray(x), float(fx)
                if verbose:
                    print(f"[iter {iteration:3d}] species {S.index} stalled -> DE refine  "
                          f"f={cscore:.3e}  ({time.time() - t_de:.2f}s)")
            else:
                cand, cscore = S.sBestPosition.copy(), S.sBestScore

            is_new = all(euclidean(cand, sp) > sep for sp, _ in solutions)
            if is_new:
                solutions.append((cand, cscore))
                if verbose:
                    print(f"[iter {iteration:3d}] species {S.index} -> NEW optimum "
                          f"({len(solutions)}/{m})  f={cscore:.3e}")
            elif verbose:
                print(f"[iter {iteration:3d}] species {S.index} hit a known optimum -> respawn")

            if len(solutions) >= m:
                S.free_members()                                    # all found -> retire
            else:
                S.respawn(bounds, obj_func, [sp for sp, _ in solutions], avoid)

    if verbose:
        print(f"RS-SPSO: {len(solutions)}/{m} solutions in {time.time() - t0:.2f}s")
    return solutions                          # already distinct by construction

In [6]:
# Himmelblau: 4 equal global minima (f=0) at
# (3,2), (-2.805,3.131), (-3.779,-3.283), (3.584,-1.848)
def himmelblau(x):
    return (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2

np.random.seed(0); random.seed(0)
bounds = [(-5, 5), (-5, 5)]
sols = rs_spso(himmelblau, bounds, m=4, N=60, max_iteration=300)

print(f"\nfound {len(sols)} distinct minima:")
for pos, score in sols:
    print(f"  x = [{pos[0]:+.4f}, {pos[1]:+.4f}]   f = {score:.2e}")

[iter  22] species 2 -> NEW optimum (1/4)  f=7.889e-31
[iter  26] species 0 -> NEW optimum (2/4)  f=0.000e+00
[iter  36] species 3 -> NEW optimum (3/4)  f=0.000e+00
[iter  37] species 1 -> NEW optimum (4/4)  f=7.889e-31

found 4 distinct minima:
  x = [-3.7793, -3.2832]   f = 7.89e-31
  x = [+3.0000, +2.0000]   f = 0.00e+00
  x = [+3.5844, -1.8481]   f = 0.00e+00
  x = [-2.8051, +3.1313]   f = 7.89e-31


## Layeb (2022) hard benchmark functions

From A. Layeb, *New hard benchmark functions for global optimization*, arXiv:2202.04606.
Verified against the paper PDF. A clean, numerically-safe subset is used here:

- **Layeb01** — unimodal, x*=1, f*=0, domain [-100,100]. Tiny optimum region in a huge flat space.
- **Layeb04** (Crossfly) — multimodal, x* alternates 0 and (2k-1)π, f*=(ln 0.001 - 1)(n-1), [-10,10].
- **Layeb11** — multimodal, x* alternates -1 and 0, f*=-(n-1), [-10,10]. Many global optima.
- **Layeb12** — multimodal, x*=2, f*=-(e+1)(n-1), [-5,5]. Paper notes CEC winners fail here.

Left out on purpose: Layeb14/15 (printed formulas hit log(0)/√(negative) → -∞/NaN, i.e. false
optima), Layeb03 (a "+1" placement ambiguity in the PDF), and the noisy Layeb19/20. Confirm those
against the author's MATLAB source before trusting them.

In [7]:
def layeb01(x):                                  # x*=1, f*=0, [-100,100]
    x = np.asarray(x)
    with np.errstate(over="ignore"):
        return np.sum(100.0 * np.sqrt(np.abs(np.exp((x - 1) ** 2) - 1.0)))


def layeb04(x):                                  # x* alt 0 & (2k-1)pi, [-10,10]
    x = np.asarray(x)
    s = 0.0
    for i in range(len(x) - 1):
        s += np.log(np.abs(x[i] * x[i + 1]) + 0.001) + np.cos(x[i] + x[i + 1])
    return s


def layeb11(x):                                  # x* alt -1 & 0, f*=-(n-1), [-10,10]
    x = np.asarray(x)
    s = 0.0
    for i in range(len(x) - 1):
        s += np.cos(x[i] * x[i + 1] + np.pi) / ((100.0 * np.abs(x[i] ** 2 - x[i + 1] - 1)) ** 2 + 1)
    return s


def layeb12(x):                                  # x*=2, f*=-(e+1)(n-1), [-5,5]
    x = np.asarray(x)
    s = 0.0
    for i in range(len(x) - 1):
        s += (np.cos(np.pi / 2 * x[i] - np.pi / 4 * x[i + 1] - np.pi / 2)
              * np.exp(np.cos(2 * np.pi * x[i] * x[i + 1])) + 1.0)
    return -s


# (name, func, bounds, m solutions to seek, known f*)
layeb_tests = [
    ("Layeb01", layeb01, [(-100, 100)] * 2, 1, 0.0),
    ("Layeb04", layeb04, [(-10, 10)] * 2, 4, np.log(0.001) - 1),
    ("Layeb11", layeb11, [(-10, 10)] * 2, 4, -1.0),
    ("Layeb12", layeb12, [(-5, 5)] * 2, 2, -(np.e + 1)),
]

np.random.seed(1); random.seed(1)
for name, f, bnds, m, fstar in layeb_tests:
    sols = rs_spso(f, bnds, m=m, N=60, max_iteration=400, verbose=False)
    print(f"\n{name}: known f* = {fstar:+.4f}   (found {len(sols)} distinct)")
    for pos, sc in sols:
        print(f"   x = [{pos[0]:+.4f}, {pos[1]:+.4f}]   f = {sc:+.4f}   |f-f*| = {abs(sc - fstar):.2e}")


Layeb01: known f* = +0.0000   (found 1 distinct)
   x = [+1.0000, +1.0000]   f = +0.0000   |f-f*| = 0.00e+00

Layeb04: known f* = -7.9078   (found 4 distinct)
   x = [-3.1416, -0.0000]   f = -7.9078   |f-f*| = 4.51e-11
   x = [+0.0000, +9.4248]   f = -7.9078   |f-f*| = 4.01e-11
   x = [+3.1416, +0.0000]   f = -7.9078   |f-f*| = 5.22e-11
   x = [-0.0000, -3.1416]   f = -7.9078   |f-f*| = 1.54e-10

Layeb11: known f* = -1.0000   (found 4 distinct)
   x = [-0.0000, -1.0000]   f = -1.0000   |f-f*| = 4.17e-13
   x = [+2.0254, +3.1022]   f = -1.0000   |f-f*| = 2.19e-12
   x = [+1.0000, +0.0000]   f = -1.0000   |f-f*| = 1.71e-12
   x = [-1.0000, -0.0000]   f = -1.0000   |f-f*| = 4.87e-14

Layeb12: known f* = -3.7183   (found 2 distinct)
   x = [-0.0000, -2.0000]   f = -3.7183   |f-f*| = 2.94e-13
   x = [+2.7913, +3.5826]   f = -3.7183   |f-f*| = 1.99e-11


## Dimension-scaling stress test

Symbols (to avoid confusion):
- **dim** = problem dimension = number of variables = `len(bounds)` (the paper's *n*)
- **N** = total particles (swarm size)
- **m** = number of distinct solutions / sub-swarms sought

Here we grow **dim** (2 → 5 → 10), scale **N** with it, keep **m** small, and report the *best*
optimum found vs the known f*. f* itself scales with dim for the pairwise functions:
Layeb04 = (ln0.001−1)(dim−1), Layeb11 = −(dim−1), Layeb12 = −(e+1)(dim−1), Layeb01 = 0.

In [8]:
# Slow (dim-10 takes ~20-30 s per function). Uncomment to run.
# def fstar_layeb(name, dim):
#     return {"Layeb01": 0.0,
#             "Layeb04": (np.log(0.001) - 1) * (dim - 1),
#             "Layeb11": -(dim - 1),
#             "Layeb12": -(np.e + 1) * (dim - 1)}[name]

# scaling = [("Layeb01", layeb01, (-100, 100)),
#            ("Layeb04", layeb04, (-10, 10)),
#            ("Layeb11", layeb11, (-10, 10)),
#            ("Layeb12", layeb12, (-5, 5))]

# np.random.seed(2); random.seed(2)
# for dim in [2, 5, 10]:
#     print(f"\n===== dim = {dim}  (N = {20 * dim}, m = 3) =====")
#     for name, f, dom in scaling:
#         bounds = [dom] * dim
#         sols = rs_spso(f, bounds, m=3, N=20 * dim, max_iteration=200, verbose=False)
#         best = min(s[1] for s in sols) if sols else float("nan")
#         fstar = fstar_layeb(name, dim)
#         print(f"  {name:8s}  best f = {best:+13.4f}   f* = {fstar:+13.4f}   error = {abs(best - fstar):.3e}")

## Swarm animation — how RS-SPSO speciates and respawns

Like the PSO animation, but the story here is **speciation + respawn**. Each particle is coloured by
its species; velocity arrows share the colour. Each species' best is a large star in its colour.
When a species finds an optimum it drops a white **X** there; if minima are still missing, that
species **respawns** — its particles jump to an unexplored region (you'll see them relocate) and
start hunting again. Species also repel each other so they spread to *different* Himmelblau minima.
With respawn there are no frozen grey particles: everyone keeps working until all `m` are found.
Trails are omitted to keep the embedded file small; `arrow_scale` shrinks arrows for readability.

In [9]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 64  # MB, so frames aren't dropped


def rs_spso_with_history(obj_func, bounds, m, N=40, r=None, c1=1.5, c2=1.5, w=0.7,
                         max_iteration=200, pos_tol=1e-4, f_tol=1e-8, patience=15,
                         k_swarm=1.0, k_particle=0.1, v_repel_max=None, sep=None,
                         refiner=default_refiner, seed=None):
    """Same algorithm as rs_spso(), but snapshots the swarm every iteration for
    animation. 2-D only. Original rs_spso() is left untouched."""
    assert len(bounds) == 2, "Animation needs a 2-D search space."
    if seed is not None:
        random.seed(seed); np.random.seed(seed)
    lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
    diag = euclidean(lo, hi)
    if r is None: r = 0.1 * diag
    if v_repel_max is None: v_repel_max = 0.1 * diag
    if sep is None: sep = 0.02 * diag
    avoid = r * 0.5
    n = max(2, N // m)

    swarm = [Particle(bounds, obj_func) for _ in range(N)]
    sorted_swarm = sorted(swarm, key=lambda p: p.pBestScore)
    seeds = select_seeds(sorted_swarm, m, r)
    subswarms = build_subswarms(sorted_swarm, seeds, n)
    solutions = []

    def snapshot(it):
        sb = np.full((m, 2), np.nan)                       # sBest per species, nan if retired
        for S in subswarms:
            if S.active:
                sb[S.index] = S.sBestPosition
        sol = np.array([p for p, _ in solutions]) if solutions else np.empty((0, 2))
        return {"it": it,
                "positions": np.array([p.position.copy() for p in swarm]),
                "velocities": np.array([p.velocity.copy() for p in swarm]),
                "swarm_ids": np.array([p.swarm_id for p in swarm]),
                "sbest": sb,
                "solutions": sol}

    history = [snapshot(0)]
    for iteration in range(max_iteration):
        active = [S for S in subswarms if S.active]
        if not active or len(solutions) >= m:
            break
        for S in active:
            for p in S.members:
                rep = repulsion_force(p, swarm, k_swarm, k_particle, v_repel_max)
                p.update_velocity(S.sBestPosition, rep, c1, c2, w)
                p.update_position(bounds); p.evaluate(obj_func)
            imp = S.update_sBest()
            S.stall_counter = S.stall_counter + 1 if imp < f_tol else 0
        for S in active:
            converged = S.radius() < pos_tol
            stalled = S.stall_counter >= patience
            if not (converged or stalled):
                continue
            if stalled and not converged:
                x, fx = refiner(obj_func, S.sBestPosition, bounds)
                cand, cscore = np.asarray(x), float(fx)
            else:
                cand, cscore = S.sBestPosition.copy(), S.sBestScore
            if all(euclidean(cand, sp) > sep for sp, _ in solutions):
                solutions.append((cand, cscore))
            if len(solutions) >= m:
                S.free_members()
            else:
                S.respawn(bounds, obj_func, [sp for sp, _ in solutions], avoid)
        history.append(snapshot(iteration + 1))
    return {"solutions": solutions, "history": history, "m": m}


def animate_rs_spso(result, bounds, obj_func, arrow_scale=0.3, interp_steps=2, interval=60):
    """Animate the swarm coloured by species, over a log-scaled objective contour.
    A species that finds an optimum drops a white X and respawns elsewhere to explore."""
    history, m = result["history"], result["m"]
    (x0, x1), (y0, y1) = bounds
    gx, gy = np.linspace(x0, x1, 240), np.linspace(y0, y1, 240)
    GX, GY = np.meshgrid(gx, gy)
    logZ = np.log1p(obj_func([GX, GY]))

    P = np.array([h["positions"] for h in history])
    V = np.array([h["velocities"] for h in history])
    SB = np.array([h["sbest"] for h in history])          # (T, m, 2)
    ids = [h["swarm_ids"] for h in history]
    sols = [h["solutions"] for h in history]
    T, N, _ = P.shape

    def interp(arr):                                       # smooth sub-frames between iterations
        out = []
        for i in range(T - 1):
            for s in range(interp_steps):
                f = s / interp_steps
                out.append((1 - f) * arr[i] + f * arr[i + 1])
        out.append(arr[-1]); return np.array(out)

    Pi, Vi, SBi = interp(P), interp(V), interp(SB)
    base = [i for i in range(T - 1) for _ in range(interp_steps)]; base.append(T - 1)
    F = len(Pi)

    palette = plt.get_cmap("tab10")(np.linspace(0, 1, 10))
    def col(sid):                                          # 0 = free (grey), else per-species colour
        return (0.6, 0.6, 0.6, 0.9) if sid == 0 else palette[(sid - 1) % 10]

    fig, ax = plt.subplots(figsize=(6.2, 5.6))
    ax.contourf(GX, GY, logZ, levels=30, cmap="viridis", alpha=0.7)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)

    c0 = np.array([col(s) for s in ids[0]])
    quiv = ax.quiver(Pi[0][:, 0], Pi[0][:, 1], Vi[0][:, 0], Vi[0][:, 1],
                     angles="xy", scale_units="xy", scale=1.0 / arrow_scale,
                     color=c0, width=0.004, alpha=0.9, zorder=4)
    scat = ax.scatter(Pi[0][:, 0], Pi[0][:, 1], c=c0, s=40,
                      edgecolors="black", linewidths=0.5, zorder=5)
    sbest = ax.scatter(SBi[0][:, 0], SBi[0][:, 1], marker="*",
                       c=[col(i + 1) for i in range(m)], s=280,
                       edgecolors="black", linewidths=0.8, zorder=6)
    found = ax.scatter([], [], marker="X", c="white", s=180,
                       edgecolors="black", linewidths=1.4, zorder=7)
    label = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top", ha="left",
                    fontsize=10, zorder=8,
                    bbox=dict(boxstyle="round", fc="white", alpha=0.85))

    def frame(t):
        b = base[t]
        cols = np.array([col(s) for s in ids[b]])
        scat.set_offsets(Pi[t]); scat.set_color(cols)
        quiv.set_offsets(Pi[t]); quiv.set_UVC(Vi[t][:, 0], Vi[t][:, 1]); quiv.set_color(cols)
        sbest.set_offsets(np.nan_to_num(SBi[t], nan=-1e9))     # nan -> push off-screen
        s = sols[b]
        found.set_offsets(s if len(s) else np.empty((0, 2)))
        label.set_text(f"iter {history[b]['it']}/{history[-1]['it']}   "
                       f"solutions found: {len(s)}/{m}")
        return [quiv, scat, sbest, found, label]

    anim = animation.FuncAnimation(fig, frame, frames=F, interval=interval, blit=True)
    plt.close(fig)                                         # avoid a duplicate static figure
    return anim


anim_bounds = [(-20, 20), (-20, 20)]
anim_result = rs_spso_with_history(himmelblau, anim_bounds, m=4, N=32,
                                   max_iteration=120, patience=12, seed=7)
anim = animate_rs_spso(anim_result, anim_bounds, himmelblau,
                       arrow_scale=0.3, interp_steps=2, interval=60)
HTML(anim.to_jshtml())

In [10]:
# --- Store & check the animation run: coverage + discovery timeline ---
himmelblau_minima = [(3.0, 2.0), (-2.805118, 3.131312),
                     (-3.779310, -3.283186), (3.584428, -1.848126)]

sols = anim_result["solutions"]
print(f"solutions stored: {len(sols)}")
for pos, sc in sols:
    print(f"   x = [{pos[0]:+.4f}, {pos[1]:+.4f}]   f = {sc:.2e}")

print("\ncoverage of the 4 known minima:")
for mx, my in himmelblau_minima:
    d = min(np.hypot(pos[0] - mx, pos[1] - my) for pos, _ in sols)
    print(f"   [{mx:+.3f}, {my:+.3f}]   nearest found = {d:.4f}   {'MISSED' if d > 0.1 else 'ok'}")

# With respawn there are no frozen particles: every particle stays in an active species
# (free count = 0) until all m minima are found. Track when each new optimum was recorded.
print("\ndiscovery timeline:")
found_prev = 0
for h in anim_result["history"]:
    found_now = len(h["solutions"])
    nfree = int(np.sum(h["swarm_ids"] == 0))
    if found_now != found_prev:
        print(f"   iter {h['it']:3d}: solutions found -> {found_now}/{anim_result['m']}   "
              f"(free particles = {nfree})")
        found_prev = found_now

solutions stored: 4
   x = [+3.5844, -1.8481]   f = 0.00e+00
   x = [-2.8051, +3.1313]   f = 7.89e-31
   x = [-3.7793, -3.2832]   f = 7.89e-31
   x = [+3.0000, +2.0000]   f = 0.00e+00

coverage of the 4 known minima:
   [+3.000, +2.000]   nearest found = 0.0000   ok
   [-2.805, +3.131]   nearest found = 0.0000   ok
   [-3.779, -3.283]   nearest found = 0.0000   ok
   [+3.584, -1.848]   nearest found = 0.0000   ok

discovery timeline:
   iter  24: solutions found -> 1/4   (free particles = 0)
   iter  32: solutions found -> 2/4   (free particles = 0)
   iter  35: solutions found -> 3/4   (free particles = 0)
   iter  48: solutions found -> 4/4   (free particles = 8)
